# E9 GNN Navigation

Author: Arush Arora

## Introduction
This codebase has mostly consisted of additive Graph Positional Encodings (GREPs) injections to provide nodal embeddings to the LLMs at hand. This new multi-stage training will rely on training an R-PEARL/GT to simply replicate the shortest-distance paths of the graph before expecting it to serve the LLM with **multiplicative** GREPs for navigation tasks, which will be factored directly into the attention-mask matrix for rendition to the LLM (as a Hadamard product cover on the attention logits). Thus, the system will be more carefully trained to incorporate the variation in model architecture among GNNs and LLMs (in terms of their pre-trained weights rather than simply their mathematical foundations).

## Mathematical Overview

## The R-PEARL GNN
The Random Positional Encoding (R-PEARL) GNN architecture is a PE generator that inputs white noise and processes it over an undirected graph $\mathcal{G} = (\mathcal{V}, \mathcal{E}, \mathcal{W})$. In this work, the graph is represented by an adjacency matrix $A$, and the GNN composes [Topology Adaptive Graph (TAG)](https://arxiv.org/abs/1710.10370) Convolutional Layers with pointwise nonlinearities (demodulators).

### Graph Convolutional Network (GNN)
The code below establishes this project's implementation of a Graph Convolutional Network, which is the foundational architecture comprising R-PEARL. The equation to demonstrate the internal architecture of this NN as follows (in most cases, $\mathbf{P}(\cdot) = \mathbf{I}(\cdot)$, where $\mathbf{I}$ is the identity function):
$$\Phi(\mathbf{X}, \mathbf{S}, \mathcal{H}) = \mathbf{X}^{(L)}$$
$$\mathbf{X}^{(0)} = \mathbf{X} \qquad \mathbf{X}^{(l)} = \mathbf{P}\Bigg[\sigma\Bigg(\sum_{k = 0}^{K^{(l)} - 1} \mathbf{S}^k\mathbf{X}^{(l - 1)}\mathbf{H}_k^{(l)}\Bigg)\Bigg]$$

### Random Graph Positional Encodings (R-PEARL)
The R-PEARL architecture extends on the GCN by instantiating it with simply one layer – a TAG Convolution and Demodulator. The mathematical equations below express the functionality of the R-PEARL network:
1. The white-noise matrix is sampled from the Gaussian distribution. $$\mathbf{Q} \in \mathbb{R}^{M \times N} \qquad \mathbf{Q} \sim \mathcal{N}(0, \mathbf{I}) \qquad \mathbf{Q} = \begin{bmatrix}
  \mathbf{q}^{(0)} & \cdots & \mathbf{q}^{(m)} & \cdots & \mathbf{q}^{(M)}
  \end{bmatrix}$$

2. The R-PEARL network has row-vector parameter $\mathbf{H}^{(0)} \in \mathbb{R}^{1 \times D}$. It takes in each column of the white-noise matrix individually and produces a sample $\mathbf{P}^{(m)} \in \mathbb{R}^{N \times D}$, which are then pooled to form GREP $\mathbf{P}$:$$\mathbf{P}^{(m)} = \Phi\Big(\mathbf{q}^{(m)};\, \mathbf{S}, \mathcal{H}\Big) = \sigma\bigg(\sum_{k = 0}^{K = 1} \mathbf{S}^k\mathbf{q}^{(m)} {\mathbf{H}}_k\bigg)$$ $$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \mathbf{P}^{(m)}$$

## Sparse Graph Transformer
The Sparse Graph Transformer (hereafter named Graph Transformer or GT) follows the same architecture as that of a normal transformer, albeit that the attention mecahnism is modified to scope only over the $k$-hop neighborhood of the query node. The mathematical equations below express the functionality of the Graph Transformer:
$$\mathbf{X}_L = \Phi\Big(\mathbf{X}_0 + \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, S, \mathcal{H})\big];\, \mathcal{T}\Big)$$
$$\mathbf{A}^{(h)}_l = \left[\frac{\exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}{\mathbf{1}^\top \exp\left[\left(\mathbf{Q}^{(h)}_l \mathbf{x}_{l-1,\ t}\right)^\top \left(\mathbf{K}^{(h)}_l \mathbf{X}_{l-1,\ U}\right)\right]}\right]^\top_{\begin{subarray}{l}t \in [N] \\[2.5pt] U = \mathcal{N}^{\le k}(t)\end{subarray}}$$
$$\mathbf{Y}^{(h)}_l = \left(\mathbf{W}_o\right)^\top_l \mathbf{V}_l \mathbf{X}_{l - 1} \left(\mathbf{A}^{(h)}_l\right)^\top$$
$$\mathbf{X}_l = \sigma\bigg(\sum_{h = 1}^H \mathbf{Y}^{(h)}_l\bigg)$$

## Transformer
The Transformer architecture follows that of the Llama3.1-8B distilled PRISM model. First, the TXT file, containing the scene-graph data, is tokenized and embedded into matrices $E$ and $\tilde{X}$ as follows, where $V$ is the size of the vocabulary and $d$ is the embedding dimension.

$$\text{TXT Tokenized Data from GPT-4: } E = \begin{bmatrix}
\mathbf{e}_1 & \mathbf{e}_2 & \overset{\mathbf{e}_t}{\cdots} & \mathbf{e}_T
\end{bmatrix}^\top \qquad \mathbf{e}_t \in \mathbb{R}^V$$

$$\text{Embed: } X = \begin{bmatrix}
\mathbf{x}_1 & \mathbf{x}_2 & \overset{\mathbf{x}_t}{\cdots} & \mathbf{x}_T
\end{bmatrix}^\top \qquad \mathbf{x}_t \in \mathbb{R}^d$$

Next, the transformer operates using the equations below:

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$$\mathbf{Z}_{1:t}^{(L)} = \operatorname{Trf}\Big(\mathbf{X}_{1:t}, {\mathcal{T}}_l\Big) \qquad {\mathcal{T}}_l = \begin{bmatrix}
\mathbf{Q}_l & \mathbf{K}_l & \mathbf{V}_l & \left(\mathbf{W}_o\right)_l
\end{bmatrix}^\top \in \mathbb{R}^{4 \times T \times D}$$

$$\hat{\mathbf{Y}}_{t + 1} = \operatorname{Linear}\Big(\mathbf{Z}_{1:t}^{(L)}\Big) \in \mathbb{R}^V$$
$$\text{Cross-Entropy Loss: } \mathcal{L}(E, \hat{\mathbf{Y}}) = \sum_t \sum_v e_{vt}\log{\hat{y}_t}$$

## Graph-Augmented LLM
The last class that is needed to create the full GREP-PRISM architecture is the `GraphAugmentedLLM`, which simply implements the following equation as a Neural Network object in PyTorch's `torch.nn` module (referring to above equations for definitions).
$$\mathbf{P} = \mathbb{\hat{E}}\Big[\mathbf{p}^{(m)}\Big] = \frac{1}{M}\sum_{m = 1}^{M} \Phi\Big(\mathbf{q}^{(m)}, \mathbf{S}, \mathcal{H}\Big)$$

$$\mathbf{X} = \mathbf{\tilde{X}} + \mathbf{P}$$

$${\mathbf{Z}}_{1:t}^{(L)} = \operatorname{Trf}\Big({\mathbf{X}}_{1:t}; \, \cdot \,\Big)$$

In [1]:
%env CUDA_VISIBLE_DEVICES=1

env: CUDA_VISIBLE_DEVICES=1


In [2]:
# Import modules.
import gc
import torch
import random
import numpy as np
import sympy as sp
import networkx as nx

from torch import nn
from itertools import product
from torch_geometric.data import Data

from prism.models.r_pearl import RandomGNNPositionalEncodings
from prism.models.gt import GraphTransformer
from prism.eval import evaluate
from prism.data import data

In [3]:
# Define a tensor rendering function.
def render_matrix(mat: torch.tensor, sig_figs: int = 0, decimals: int = 0):
    out = sp.Matrix(mat.detach().cpu().numpy())
    if sig_figs > 0:
        return sp.N(out, sig_figs)
    if decimals > 0:
        return out.applyfunc(lambda x: x.round(decimals))
    return out

In [4]:
# Standard options.
eval_path = '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
device = 'cuda'

In [5]:
# Setup eval infrastructure.
samples_by_graph, graph_file_by_name = data.load_samples_by_graph(eval_path)
graph_file = random.choice(list(samples_by_graph.keys()))
eval_data = samples_by_graph[graph_file]
eval_data = {graph_file: [random.choice(eval_data)]}

In [6]:
print(f'/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/{graph_file}.html')

/Users/cyberlives/Documents/GitHub/GREP-PRISM/eval/render/revised/data_gen_019.html


## Experiments

### §1 Pretraining a GNN to Classify Edge Existence
We first hope to optimize a GNN (R-PEARL or Graph Transformer) to classify whether an edge exists in the graph or not. Such a model will serve as a backbone pretrained model for fine-tuning on reporting shortest paths. The equations to represent this procedure are below:
$$H = \mathbb{\hat{E}}_{\mathbf{q}}\big[\Phi(\mathbf{q};\, \cdot\,)\big]$$
$$\mathbf{\hat{y}}_{ij} = \text{MLP}\big[\mathbf{h}_i\ ||\ \mathbf{h}_j\big] \in [0, 1]$$

In [7]:
# Instantiate a GNN.
model_type = 'gt'
if model_type == 'gt':
    gnn = GraphTransformer(
        num_layers=3,
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        heads=8,
        num_samples=320,
        dropout=0.1,
        k_pe=3,
        k_gt=2,
        eps=1e-6,
        use_layer_norm=True
    )
    gnn.out_features = gnn.d_model
else:
    gnn = RandomGNNPositionalEncodings(
        pe_hidden_channels=256,
        pe_num_layers=5,
        d_model=1024,
        num_samples=320,
        dropout=0.1,
        k=3,
        eps=1e-6,
        use_layer_norm=True
    )
    gnn.out_features = gnn.d_model

In [ ]:
from typing import Union


# Define a class for edge detection and instantiate it.
class GNNEdgeDetector(nn.Module):
    """
    Simple class to detect whether Node 1 and Node 2 are connected by
    applying an MLP to the Graph Positional Encodings of both nodes concatenated.
    """
    def __init__(self, gnn: Union[RandomGNNPositionalEncodings, GraphTransformer]):
        super(GNNEdgeDetector, self).__init__()
        shape = gnn.out_features
        self.classifier = nn.Sequential(
            nn.Linear(4 * shape, shape),
            nn.LeakyReLU(),
            nn.Linear(shape, 1)
        )
        self.graph = Data(
            x=torch.empty((0, 0), dtype=torch.float),
            edge_index=torch.empty((2, 0), dtype=torch.long)
        )
        self.cached_pe = torch.zeros(size=(1, shape))
        self.gnn = gnn

    def forward(self, graph: Data, node1: int, node2: int):
        if self.cached_pe is None or not self.cached_pe.any() or self.graph is not graph:
            self.graph = graph
            self.cached_pe = self.gnn(self.graph)
        hi, hj = self.cached_pe[node1], self.cached_pe[node2]
        return self.classifier(torch.cat((abs(hi), abs(hj), hi * hj, abs(hi - hj)), dim=0))
    
    def invalidate_cache(self):
        self.graph = None
        self.cached_pe = None


# Instantiate the class.
detector = GNNEdgeDetector(gnn)

#### Numeric Visualizations with SymPy
Using the `render_matrix()` function defined at the very beginning of this notebook, we first explore the procedure needed to preprocess a pre-training set for the GNN to reconstruct the eigenbasis of the graph adjacency given a scene graph PyTorch `Data` object.

In [9]:
# Prepare a graph from the data to be used in the GNN.
from torch_geometric.utils import to_dense_adj, to_networkx
from prism.data import utils
import numpy as np
import sympy as sp

# Prepare the graph for rendition.
ex_graph = utils.scene_graph_dict_to_pyg(eval_data[graph_file][0][2])
adj = to_dense_adj(ex_graph.edge_index).squeeze().cuda()

# Show the adjacency matrix of the graph.
render_matrix(adj)

Matrix([
[  0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0, 1.0,   0,   0,   0,   0,   0,   0,   0,   0,   0],
[  0,   0,   0,   0,   0,   0,   0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0,   0,   0

In [10]:
# Feed the matrix to the GNN.
out = gnn(ex_graph).to(device)
_, _, V = torch.pca_lowrank(out, q=10, center=True)
out = out - out.mean(dim=0)
render_matrix(out @ V, 3)

Matrix([
[  5.21, -0.229,    1.24,   1.06,  0.853, -0.879,  0.841,   0.0671,   -1.21,   -2.86],
[  4.53,  0.314,   0.244,   2.07,   1.35,   2.33,   1.17,     1.43,    1.83,  -0.567],
[   7.1,  -0.79,  -0.182,    0.7,   -1.6,  -0.37, -0.648,    -1.41,   0.425,   -1.12],
[   7.7,   1.25,  -0.826,  -2.73,   1.56,   1.21, -0.259,    -1.33,  -0.622,    1.02],
[  3.05, -0.995,   -1.48,   1.82,   -1.8,  -1.97,   1.57,     3.26,   -2.39,  -0.418],
[  6.92,  -1.31,   -1.67,  -1.96,  0.265,   -3.0,  0.734,     1.08,   -1.33,   0.896],
[ -4.99,  0.167,   -2.35,  0.509,   2.79,  -1.65,   2.58,     1.24, -0.0214,    3.59],
[ 0.556,   0.67,  -0.835,  -1.97,  -2.02,   1.85,  -1.46,   -0.408,   0.477,   0.443],
[  -3.4,  -2.48,   -1.61,  -1.36,  -2.24, -0.769,   2.21,    -2.54,   -1.03,   -1.13],
[ -2.65,  -0.31,    5.87,  -2.09, -0.537,   -3.7,   1.08,   -0.444,  -0.313,    1.32],
[  1.09,  -1.09,    1.25,  -1.82,  -2.72, -0.385,  -1.05,   -0.984,    2.57,   0.601],
[  0.77,   1.11,   0.142,   1.31, 

In [11]:
# Test out the Detector.
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
out = detector(ex_graph, node1, node2).to(device)
render_matrix(out, 3)

Matrix([[-0.414]])

#### Pre-Training of GNN on Edge Incidence
Next, we actually preprocess and train the GNN using the steps defined above.

In [22]:
# Import Modules.
from torch_geometric.loader import DataLoader

# Configure the training and test datasets.
train_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/train_graphs'
)
test_dataset, _ = data.load_samples_by_graph(
    '../data/revised/gen/nav100_n30_gemma_data/split/test_graphs'
)

# Configure the validation dataset.
train_prop = 0.8
train_num = len(train_dataset)
train_keys = random.sample(list(train_dataset.keys()), k=int(train_num * train_prop))
val_dataset = {k: v for k, v in train_dataset.items() if k not in train_keys}
train_dataset = {k: v for k, v in train_dataset.items() if k in train_keys}

# Add edge existence tuples for edge existence.
def generate_data(dataset):
    graphs = [utils.scene_graph_dict_to_pyg(v[0][2]) for _, v in dataset.items()]
    for graph in graphs:
        graph.x = graph.x.to(device)
        graph.edge_index = graph.edge_index.to(device)
        combs = torch.tensor(
            [[u, v] for u in range(graph.num_nodes) for v in range(u + 1, graph.num_nodes)],
            device=device
        ).T
        existence = torch.tensor(
            [combs[:, i].tolist() in graph.edge_index.T.tolist() for i in range(combs.shape[1])],
            device=device
        )
        exclusion = combs[:, ~existence].to(device)
        indices = torch.randint(
            high=exclusion.shape[1], size=(graph.edge_index.shape[1],), device=device
        )
        graph.edges_x = torch.cat((graph.edge_index, exclusion[:, indices]), dim=1).to(device)
        graph.edges_y = torch.cat(
            (torch.ones(size=(graph.edge_index.shape[1],)), torch.zeros(size=(indices.shape[0],))), 
            dim=0
        ).to(device)
    
    return graphs


train_graphs = generate_data(train_dataset)
val_graphs = generate_data(val_dataset)
test_graphs = generate_data(test_dataset)

In [13]:
# Train the GNN to reconstruct the eigenvectors of the graph adjacency.
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch_geometric.loader import DataLoader

batch_size = 4
val_freq = 5
epochs = 50

def test_loop(dataloader, model, loss_fn):
    model.to(device)
    model.eval()
    size = len(dataloader.dataset)
    num_samples = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for graph in dataloader.dataset:
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            test_loss += loss_fn(preds, graph.edges_y).item()
            correct += ((preds.sigmoid() > 0.5).float() == graph.edges_y).float().mean().item()

    test_loss /= num_samples
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    return test_loss


def train_loop(train_dataloader, val_dataloader, model, loss_fn, 
               optimizer, scheduler, batch_size=20, epochs=50):
    size = len(train_dataloader.dataset)
    model.to(device)
    model.train()
    val_loss: float = 0
    for i in range(epochs):
        # Validation loop.
        if i % val_freq == 0:
            print(f"=============\nValidation #{i // val_freq}\n=============")
            val_loss = test_loop(val_dataloader, model, loss_fn)
            model.train()
        
        print(f"=============\nEpoch #{i}\n=============")
        for j, graph in enumerate(train_dataloader.dataset):
            # Compute prediction and loss.
            optimizer.zero_grad()
            preds = torch.stack([
                model(graph, graph.edges_x[0, k], graph.edges_x[1, k])
                for k in range(graph.edges_x.shape[1])
            ]).squeeze(-1).to(device)
            loss = loss_fn(preds, graph.edges_y)

            # Backpropagation.
            loss.backward()
            optimizer.step()
            model.invalidate_cache()
            if scheduler:
                scheduler.step(val_loss)

            # Results.
            if j % batch_size == 0:
                loss, current = loss.item(), j
                print(f"Loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


loss_fn = nn.BCEWithLogitsLoss()
train_dataloader = DataLoader(train_graphs, batch_size=batch_size)
val_dataloader = DataLoader(val_graphs, batch_size=batch_size)
optimizer = torch.optim.AdamW(detector.parameters(), lr=3e-5)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)
train_loop(train_dataloader, val_dataloader, detector, loss_fn, 
           optimizer, None, batch_size=batch_size, epochs=epochs)

Validation #0
Test Error: 
 Accuracy: 50.0%, Avg loss: 2.857612 

Epoch #0
Loss: 0.714955  [    0/   32]
Loss: 0.725419  [    4/   32]
Loss: 0.697065  [    8/   32]
Loss: 0.652811  [   12/   32]
Loss: 0.631417  [   16/   32]
Loss: 0.592692  [   20/   32]
Loss: 0.629228  [   24/   32]
Loss: 0.664077  [   28/   32]
Epoch #1
Loss: 0.604740  [    0/   32]
Loss: 0.613513  [    4/   32]
Loss: 0.611979  [    8/   32]
Loss: 0.616696  [   12/   32]
Loss: 0.606340  [   16/   32]
Loss: 0.562413  [   20/   32]
Loss: 0.604812  [   24/   32]
Loss: 0.650255  [   28/   32]
Epoch #2
Loss: 0.568710  [    0/   32]
Loss: 0.564407  [    4/   32]
Loss: 0.627094  [    8/   32]
Loss: 0.561592  [   12/   32]
Loss: 0.537348  [   16/   32]
Loss: 0.514801  [   20/   32]
Loss: 0.611714  [   24/   32]
Loss: 0.634957  [   28/   32]
Epoch #3
Loss: 0.525419  [    0/   32]
Loss: 0.521642  [    4/   32]
Loss: 0.588852  [    8/   32]
Loss: 0.538991  [   12/   32]
Loss: 0.509485  [   16/   32]
Loss: 0.504745  [   20/   32

In [14]:
torch.save(detector, '../outputs/e9_multistage_training/edge_detector_2.pt')
torch.save(detector.gnn.state_dict(), f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt')
# gnn.load_state_dict(torch.load(f'../outputs/e9_multistage_training/edge_detector_{model_type}.pt'))

#### Evaluation of Pre-Trained GNN on Edge Incidence
We now test the trained model on the evaluation dataset. First, we we will render the output for clarity.

In [35]:
# Test out the Detector.
node1, node2 = random.sample(range(ex_graph.num_nodes), k=2)
# out = detector(ex_graph, node1, node2).to(device)
# render_matrix(out, 3)

In [38]:
# Evaluate the GNN on its reconstruction of eigenvectors of test graph adjacencies.
test_dataloader = DataLoader(test_graphs, batch_size=5)
test_loop(test_dataloader, detector, loss_fn)

Test Error: 
 Accuracy: 50.1%, Avg loss: 3.615018 



3.615017682313919